## **CÀI ĐẶT THƯ VIỆN**

In [1]:
!pip install faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/18.8 MB ? eta -:--:--

   ━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.5/18.8 MB 17.0 MB/s eta 0:00:02

   ━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━ 10.5/18.8 MB 126.2 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━ 17.1/18.8 MB 186.7 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 18.8/18.8 MB 191.1 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 75.3 MB/s eta 0:00:00


In [2]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!mv cloudflared-linux-amd64 cloudflared
!./cloudflared --version

cloudflared version 2026.7.3 (built 2026-07-23-09:58 UTC)


## **IMPORT THƯ VIỆN**

In [3]:
import torch

import numpy as np
import faiss
import os
from tqdm.notebook import tqdm
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print("DEVICE: ", device)

DEVICE:  cuda


## **LOADMODEL**

#### Load Align 

In [4]:
from transformers import AlignProcessor, AlignModel

def loadAlign():
    # ~ 900 MB 
    processor = AlignProcessor.from_pretrained("kakaobrain/align-base")
    model = AlignModel.from_pretrained("kakaobrain/align-base").to(device)
    model.eval()
    return model, processor


#### Load MetaCLIP2

In [5]:
MODEL_NAME = "facebook/metaclip-2-worldwide-giant-378"

def load_metaclip2():
    processor = AutoProcessor.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, attn_implementation="sdpa").to(device)
    model.eval()
    return model, processor

#### Load SigLIP2

In [6]:
import torch
import torch.nn.functional as F

from transformers import AutoProcessor, AutoModel

def load_siglip2(model_name="google/siglip2-so400m-patch14-384", device=device):
    processor = AutoProcessor.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()
    return model, processor

In [7]:
# # Load mo hinh
# align_model, align_processor = loadAlign()

In [8]:
metaclip2_model, metaclip2_processor = load_metaclip2()

preprocessor_config.json:   0%|          | 0.00/515 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/18.2M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1294 [00:00<?, ?it/s]

In [9]:
siglip2_model, siglip2_processor = load_siglip2()

preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/559 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

## **BACKEND** 

In [10]:
class FaissWrapper:
    
    def __init__(self, faiss_index, ids):
        self.faiss_index = faiss_index
        self.ids = list(ids) if not isinstance(ids, list) else ids
        self.imgId_to_ids = {imgId : idx for idx, imgId in enumerate(self.ids)}
        
    def search(self, vector, k=1000):
        D, I = self.faiss_index.search(vector, k)
        
        I = I[0]
        D = D[0]

        img_ids = [self.ids[i] for i in I if i >=0]

        return img_ids, D

    def get_internal_vector(self, imgId):
            
        faiss_internal_id = self.imgId_to_ids[imgId]
        vector = self.faiss_index.reconstruct(faiss_internal_id)
        vector = np.atleast_2d(vector)

        return vector

In [11]:
from abc import ABC, abstractmethod

class BaseExtractor(ABC):
    def __init__(
        self,
        model,
        processor,
        device = device
    ):
        self.model = model
        self.processor = processor
        self.device = device
        
    @abstractmethod
    def get_text_feature(self, text: str):
        pass

    
    def get_image_feature(self, img):
        pass


class AlignExtractor(BaseExtractor):
    
    def get_text_feature(self, text: str):
        
        with torch.no_grad():
            inputs = self.processor(text=text, return_tensors="pt").to(device)
    
            text_embeds = self.model.get_text_features(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                token_type_ids=inputs['token_type_ids'],
            )
            
            text_embeds = F.normalize(text_embeds.pooler_output, dim=-1).cpu().numpy()
        
        return np.atleast_2d(text_embeds)



class MetaCLIP2Extractor(BaseExtractor):
    
    def get_text_feature(self, text: str):
        
        with torch.no_grad():
            inputs = self.processor(text=text, return_tensors="pt").to(device)
    
            text_embeds = self.model.get_text_features(
                **inputs
            )
            
            text_embeds = F.normalize(text_embeds.pooler_output, dim=-1).float().cpu().numpy()
        
        return np.atleast_2d(text_embeds)

    def get_image_feature(self, img):
        with torch.no_grad():
            inputs = self.processor(
                                images=img,
                                return_tensors="pt"
                            ).to(device)
                    
            features_obj = self.model.get_image_features(**inputs)
            features = features_obj.pooler_output
            features = F.normalize(features, dim=-1).float().cpu().numpy()
    
        return np.atleast_2d(features)
         

class SigLIP2Extractor(BaseExtractor):
    
    def get_text_feature(self, text: str):
        
        with torch.no_grad():
            inputs = self.processor(text=text, return_tensors="pt",  padding="max_length", max_length=64, truncation=True).to(device)
    
            text_embeds = self.model.get_text_features(
                **inputs
            )
            
            text_embeds = F.normalize(text_embeds.pooler_output, dim=-1).cpu().numpy()
        
        return np.atleast_2d(text_embeds)

    def get_image_feature(self, img):
        with torch.no_grad():
            inputs = self.processor(
                                images=img,
                                return_tensors="pt"
                            ).to(device)
                    
            features_obj = self.model.get_image_features(**inputs)
            features = features_obj.pooler_output
            features = F.normalize(features, dim=-1).cpu().numpy()
    
        return np.atleast_2d(features)

In [12]:
# align_extractor = AlignExtractor(align_model, align_processor)


metaclip2_extractor = MetaCLIP2Extractor(metaclip2_model, metaclip2_processor)

siglip2_extractor = SigLIP2Extractor(siglip2_model, siglip2_processor)

In [13]:
loaded_indices = {}

def load_index(feature_type):
    
    if feature_type in loaded_indices:
        return loaded_indices[feature_type]

    index_wrapper = None
    index_path = f'/kaggle/input/datasets/nhnguynvn/faiss-batch1/faiss-index_{feature_type}.faiss'
    idmap_path = f'/kaggle/input/datasets/nhnguynvn/faiss-batch1/faiss-idmap_{feature_type}.txt'
    
    if os.path.exists(index_path) and os.path.exists(idmap_path):
        # read faiss index
        index = faiss.read_index(index_path, faiss.IO_FLAG_MMAP)
        # read idmap
        with open(idmap_path, 'r') as lines:
            ids = list(map(str.rstrip, lines))

        index_wrapper = FaissWrapper(index, ids)
        loaded_indices[feature_type] = index_wrapper

    return index_wrapper

In [14]:
features_type = ['siglip2', 'metaclip2']
for t in features_type:
    load_index(t)

print('DANH SACH INDEX: ', loaded_indices.keys())

DANH SACH INDEX:  dict_keys(['siglip2', 'metaclip2'])


In [15]:
from pydantic import BaseModel,  HttpUrl, Base64Bytes

class TextualQuery(BaseModel):
    textual: str
    mode: str
    k: int = 1000

class VfQuery(BaseModel):
    imgId: str
    k: int = 1000

class QbeQuery(BaseModel):
    image_input: HttpUrl | Base64Bytes | str | None  = None
    k: int = 1000

In [16]:
from io import BytesIO
from pathlib import Path
import base64

import requests
from PIL import Image


def load_image(image_input):
    """
    image_input có thể là:
        - PIL.Image.Image
        - local path
        - http/https url
        - bytes
        - base64 string
    """

    # PIL Image
    if isinstance(image_input, Image.Image):
        return image_input.convert("RGB")

    # bytes
    if isinstance(image_input, bytes):
        return Image.open(BytesIO(image_input)).convert("RGB")

    # string
    if isinstance(image_input, str):

        # URL
        if image_input.startswith(("http://", "https://")):
            response = requests.get(image_input, timeout=10)
            response.raise_for_status()

            return Image.open(BytesIO(response.content)).convert("RGB")

        # Base64
        if image_input.startswith("data:image"):
            image_input = image_input.split(",", 1)[1]

        try:
            decoded = base64.b64decode(image_input, validate=True)
            return Image.open(BytesIO(decoded)).convert("RGB")
        except Exception:
            pass

        # Local path
        path = Path(image_input)
        if path.exists():
            return Image.open(path).convert("RGB")

    raise ValueError("Unsupported image input")

In [17]:
from fastapi import FastAPI


app = FastAPI()

@app.get("/ping")
async def root():
    return {"message": "Hello from Kaggle"}



@app.post('/textual')
async def search_textual(query: TextualQuery):
    textual = query.textual
    mode = query.mode
    k = query.k
    
    # if mode == "all":
    #     feat_align = align_extractor.get_text_feature(textual)
    #     feat_siglip2 = siglip2_extractor.get_text_feature(textual)
    #     feat_openclip = clip_extractor.get_text_feature(textual)

    #     img_ids, scores = loaded_indices['siglip2'].search(feat_siglip2, k)
    #     response = [
    #         {
    #            "score": round(float(score), 3),
    #             "imgId": img_id,
    #             "videoId": img_id.split('-')[0],
    #             "selectedFrame": int(img_id.split('-')[1])
                
    #         }
    #             for (img_id, score) in zip(img_ids, scores)
    #     ]
    #     return response
        
    if mode == 'metaclip2':
        feat = metaclip2_extractor.get_text_feature(textual)
        
    elif mode == "siglip2":
        feat = siglip2_extractor.get_text_feature(textual)
        
    img_ids, scores = loaded_indices[mode].search(feat, k)
    response = [
        {
            "score": round(float(score), 3),
            "imgId": img_id,
            "videoId": img_id.split('-')[0],
            "selectedFrame": int(img_id.split('-')[1])
            
        }
            for (img_id, score) in zip(img_ids, scores)
    ]
    return response
        
    
@app.post("/vf")
async def search_vf(vf_query: VfQuery):
    imgId = vf_query.imgId
    k = vf_query.k
    
    siglip2_index = loaded_indices['siglip2']
    internal_vector = siglip2_index.get_internal_vector(imgId)

    img_ids, scores = siglip2_index.search(internal_vector, k)
    
    response = [
        {
            "score": round(float(score), 3),
            "imgId": img_id,
            "videoId": img_id.split('-')[0],
            "selectedFrame": int(img_id.split('-')[1])
            
        }
            for (img_id, score) in zip(img_ids, scores)
    ]
    return response

@app.post("/qbe")
async def search_qbe(query: QbeQuery):
    image_input = query.image_input
    k = query.k
    
    img = load_image(image_input)

    siglip2_index = loaded_indices['siglip2']

    feat = siglip2_extractor.get_image_feature(img)
    
    img_ids, scores = siglip2_index.search(feat, k)
    response = [
        {
            "score": round(float(score), 3),
            "imgId": img_id,
            "videoId": img_id.split('-')[0],
            "selectedFrame": int(img_id.split('-')[1])
            
        }
            for (img_id, score) in zip(img_ids, scores)
    ]
    return response



In [18]:
import uvicorn
import threading
import nest_asyncio

nest_asyncio.apply()

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run, daemon=True).start()

In [19]:
import subprocess
import re

proc = subprocess.Popen(
    ["./cloudflared","tunnel","--url","http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in proc.stdout:
    print(line, end="")
    m = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
    if m:
        print("PUBLIC URL:", m.group(0))
        break

INFO:     Started server process [23]


INFO:     Waiting for application startup.


INFO:     Application startup complete.


INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


2026-08-11T13:26:50Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-11T13:26:50Z INF Requesting new quick Tunnel on trycloudflare.com...


2026-08-11T13:26:55Z INF +--------------------------------------------------------------------------------------------+
2026-08-11T13:26:55Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-08-11T13:26:55Z INF |  https://template-forge-single-hold.trycloudflare.com                                      |
PUBLIC URL: https://template-forge-single-hold.trycloudflare.com


In [20]:
import time
from IPython.display import clear_output

while True:
    time.sleep(240)  # mỗi 4 phút
    clear_output(wait=True)
    print("keepalive:", time.strftime("%H:%M:%S"))

keepalive: 15:18:55
